# CSE 151B — Full Pipeline (External Math SFT + DPO + GRPO + Self-Consistency)

End-to-end recipe assuming **unlimited compute time**. Ranked by expected gain:

1. **QLoRA SFT on a large external math corpus** (biggest lever for a 4B model).
2. **DPO** on self-sampled (correct, incorrect) pairs from the SFT model.
3. **GRPO** (optional, last) with the project `Judger` as the outcome reward.
4. **Self-consistency** with large N at inference time.
5. Few-shot + CoT (always on) and **progressive-hint** for low-confidence items.
6. Greedy CoT fallback (the implicit baseline).

## Recoverable checkpoints
Every long stage writes incremental JSONL/Trainer checkpoints. Re-running a cell after a crash **resumes** instead of starting over:

| Stage | Resumable artifact |
|---|---|
| External SFT data conversion | `training/external_sft.jsonl` — re-runs skip if file exists |
| QLoRA SFT training | `training/sft_lora/checkpoint-*` (HF Trainer `resume_from_checkpoint=True`) |
| Self-sampling on public set | `training/self_samples.jsonl` — appended per batch, dedup by `(id, sample_idx)` |
| DPO pair construction | `training/dpo_pairs.jsonl` — idempotent rebuild from samples |
| DPO training | `training/dpo_lora/checkpoint-*` |
| GRPO training | `training/grpo_lora/checkpoint-*` |
| Private inference | `results/private_samples.jsonl` — appended per chunk, skip done ids on resume |
| Progressive-hint round | `results/private_hint_samples.jsonl` — same |

## External datasets (all on Hugging Face Hub, free, publicly available)
Listed best-to-worst for this task. The notebook auto-downloads them via `datasets.load_dataset`. You need `huggingface_hub` logged in only if you want gated datasets — none of these require auth.

| HF dataset id | Size | Why | License |
|---|---|---|---|
| `AI-MO/NuminaMath-CoT` | ~860k | High-quality competition-math problems w/ CoT solutions, the strongest single corpus for math reasoning SFT in 2024–25. | Apache-2.0 |
| `nvidia/OpenMathInstruct-2` | ~14M | Massive synthetic math QA w/ solutions distilled from Llama-3.1-405B; broad coverage. | CC-BY-4.0 |
| `meta-math/MetaMathQA` | ~395k | Rephrased GSM8K + MATH; great for arithmetic + word problems. | MIT |
| `TIGER-Lab/MathInstruct` | ~260k | Mix of CoT + program-of-thought; helps with symbolic problems. | MIT |
| `hendrycks/competition_math` | ~12.5k | The MATH benchmark train split; gold-standard solutions. | MIT |
| `openai/gsm8k` (`main` config) | ~7.5k | Grade-school word problems; easy wins. | MIT |
| `nvidia/OpenMathReasoning` | ~3M | Reasoning-traces dataset (uses Qwen/DeepSeek teacher); aligns well with Qwen3-Thinking style. | CC-BY-4.0 |

For a single overnight run start with **NuminaMath-CoT + MetaMathQA + competition_math + gsm8k** (~1.3M rows). Add OpenMathInstruct-2 only if you have multi-day budget.

Run the cells **top to bottom**. Each stage's switch (`RUN_*`) defaults to the recommended setting.

## 0. Environment
Reuses the project `.venv`. First run uncomment the pip line.

In [1]:
# One-time install (uncomment first run).
# !.venv/bin/python -m pip install -q --upgrade peft trl datasets accelerate sentencepiece
!source ./.venv/bin/activate && echo "venv ready"

venv ready


## 1. Configuration
Edit the switches and dataset list here. Defaults assume you want every stage on.

In [ ]:
import os, json, re, sys, gc, csv, math, random, time, glob, shutil, hashlib
from pathlib import Path
from typing import Optional, List, Tuple, Iterable
from collections import Counter, defaultdict

BASE_MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"

PUBLIC_PATH      = "data/public.jsonl"
PRIVATE_PATH     = "data/private.jsonl"

RESULTS_DIR      = Path("results");  RESULTS_DIR.mkdir(exist_ok=True)
TRAIN_DIR        = Path("training"); TRAIN_DIR.mkdir(exist_ok=True)

EXT_SFT_PATH          = TRAIN_DIR / "external_sft.jsonl"          # single-answer math
EXT_SFT_MCQ_PATH      = TRAIN_DIR / "external_sft_mcq.jsonl"      # MCQ math (NEW)
MULTI_ANSWER_SFT_PATH = TRAIN_DIR / "multi_answer_sft.jsonl"      # synth multi-[ANS] (NEW)
SELF_SAMPLES_PATH     = TRAIN_DIR / "self_samples.jsonl"
DPO_PAIRS_PATH        = TRAIN_DIR / "dpo_pairs.jsonl"
GRPO_PROMPTS_PATH     = TRAIN_DIR / "grpo_prompts.jsonl"
SFT_LORA_DIR          = TRAIN_DIR / "sft_lora"
DPO_LORA_DIR          = TRAIN_DIR / "dpo_lora"
GRPO_LORA_DIR         = TRAIN_DIR / "grpo_lora"
MERGED_SFT_DIR = Path(f"/tmp/{os.environ['USER']}/151b_training/merged_sft")
MERGED_SFT_DIR.parent.mkdir(parents=True, exist_ok=True)
MERGED_FINAL_DIR = Path(f"/tmp/{os.environ['USER']}/151b_training/merged_final")
MERGED_FINAL_DIR.parent.mkdir(parents=True, exist_ok=True)
PRIVATE_SAMPLES_PATH = RESULTS_DIR / "private_samples.jsonl"
PRIVATE_HINTS_PATH   = RESULTS_DIR / "private_hint_samples.jsonl"
PRIVATE_SUBPART_PATH = RESULTS_DIR / "private_subpart_samples.jsonl"
SUBMISSION_CSV       = Path("submission.csv")

# ── Stage switches ──────────────────────────────────────────────────────────
RUN_BUILD_EXT_SFT       = False  # single-answer math corpus (existing)
RUN_BUILD_EXT_MCQ       = False  # MCQ corpus (NEW — flip to build MMLU math subsets)
RUN_BUILD_MULTI_ANSWER  = False  # synthetic multi-[ANS] corpus (NEW — flip to synthesize)
RUN_SFT                 = False  # train QLoRA on all available SFT corpora
RUN_MERGE_SFT           = True
RUN_SELF_SAMPLE         = False
RUN_BUILD_DPO           = False
RUN_DPO                 = False
RUN_GRPO                = False
RUN_MERGE_FINAL         = True
RUN_PRIVATE_INFER       = True
RUN_HINT_ROUND          = True
RUN_SUBPART_FALLBACK    = True   # option D — runs after voting+hint, multi-[ANS] only

# ── External SFT corpora ────────────────────────────────────────────────────
# Math single-answer / CoT — primary reasoning corpus.
EXT_DATASETS = [
    ("AI-MO/NuminaMath-CoT",        None,   50_000),
    ("meta-math/MetaMathQA",         None,   30_000),
    ("hendrycks/competition_math",   None,     None),
    ("openai/gsm8k",                 "main",   None),
]

# MCQ math — bridges the format gap for letter-style \boxed{A} answers.
# MMLU is a benchmark; using subsets as SFT data is fine because it isn't the
# competition's evaluation set. Each row: question, choices[4], answer (idx).
EXT_DATASETS_MCQ = [
    ("cais/mmlu", "high_school_mathematics", None),
    ("cais/mmlu", "college_mathematics",     None),
    ("cais/mmlu", "elementary_mathematics",  None),
    ("cais/mmlu", "abstract_algebra",        None),
    ("cais/mmlu", "high_school_statistics",  None),
    ("cais/mmlu", "formal_logic",            None),
    ("cais/mmlu", "high_school_physics",     None),
]

# Synthetic multi-[ANS] — concatenates K∈[MIN,MAX] short single-answer problems
# into one multi-slot problem so the model learns to emit \boxed{a, b, c}.
N_MULTI_ANSWER         = 8_000
MULTI_ANSWER_MIN_SLOTS = 2
MULTI_ANSWER_MAX_SLOTS = 5

# ── Hyperparameters ─────────────────────────────────────────────────────────
GPU_ID                = "0"
MAX_MODEL_LEN         = 8192
INFER_MAX_MODEL_LEN   = 16384
MAX_NEW_TOKENS_INFER  = 8192
MAX_NEW_TOKENS_SAMPLE = 4096

N_SAMPLES_PER_Q       = 16
N_SAMPLES_SELF        = 6
PROGRESSIVE_HINT_TAU  = 0.5
N_HINT_SAMPLES        = 8
FEW_SHOT_K_MCQ        = 2
FEW_SHOT_K_FREE       = 2

# Option C — stratified inference sampling (precision + exploration).
STRATIFIED_SAMPLING   = True
INFER_T_PRECISION     = 0.3
INFER_T_EXPLORATION   = 0.9
INFER_TEMPERATURE     = 0.7
INFER_TOP_P           = 0.95
INFER_TOP_K           = 20
INFER_CHUNK           = 64

# Option D — per-subpart fallback.
SUBPART_CONF_THRESHOLD = 0.3
N_SUBPART_SAMPLES      = 4
SUBPART_TEMP           = 0.5

SFT_EPOCHS            = 2          # 2 epochs over the combined corpus (bumped from 1)
SFT_LR                = 1e-4
SFT_BSZ               = 1
SFT_GRAD_ACCUM        = 8
SFT_SAVE_STEPS        = 200
SFT_LOG_STEPS         = 20

DPO_EPOCHS            = 1
DPO_LR                = 5e-6
DPO_BETA              = 0.1

GRPO_EPOCHS           = 1
GRPO_LR               = 5e-6
GRPO_NUM_GENERATIONS  = 4
GRPO_BETA             = 0.04

os.environ["CUDA_VISIBLE_DEVICES"]        = GPU_ID
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_USE_DEEP_GEMM"]      = "0"
os.environ["VLLM_DEEP_GEMM_WARMUP"]   = "skip"
os.environ["TOKENIZERS_PARALLELISM"]      = "false"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

random.seed(0)
print("Config OK.")


## 2. Load competition data + judger

In [4]:
sys.path.insert(0, ".")
from judger import Judger
from utils import last_boxed_only_string, remove_boxed
judger = Judger(strict_extract=False)

def load_jsonl(p):
    return [json.loads(l) for l in open(p)]

public_data  = load_jsonl(PUBLIC_PATH)
private_data = load_jsonl(PRIVATE_PATH)
print(f"public={len(public_data)}  private={len(private_data)}")

public=1126  private=943


## 3. Prompt construction (CoT + few-shot + optional progressive hint)

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Reason carefully and step-by-step. "
    "At the very end, output ONLY the final answer wrapped in a single \\boxed{}. "
    "If the problem has multiple [ANS] placeholders, output ALL final answers in order, "
    "comma-separated, inside ONE \\boxed{}, e.g. \\boxed{3, 7, -1}. "
    "Do not add units inside \\boxed{} unless the question explicitly asks. "
    "Simplify radicals and fractions where possible."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and the answer choices, reason step-by-step, "
    "then output ONLY the letter of the single best option inside \\boxed{}, e.g. \\boxed{C}."
)
SYSTEM_PROMPT_SLOT = (
    "You are an expert mathematician. The question has multiple answer slots labeled "
    "[ANS_1], [ANS_2], etc. Solve ONLY the slot you are asked about. Show your reasoning "
    "step-by-step, then output the value for that single slot inside one \\boxed{}."
)

def _short(item, lim_q=350, lim_a=80):
    if len(item["question"]) > lim_q: return False
    if item.get("options"):
        return len(str(item.get("answer", ""))) <= 3 and all(len(o) <= lim_a for o in item["options"])
    a = item.get("answer", [])
    if not isinstance(a, list): a = [a]
    return len(a) <= 2 and all(len(str(x)) <= 30 for x in a)

_mcq_short  = [d for d in public_data if d.get("options") and _short(d)]
_free_short = [d for d in public_data if not d.get("options") and _short(d)]
random.Random(7).shuffle(_mcq_short); random.Random(7).shuffle(_free_short)
FEW_SHOT_MCQ  = _mcq_short[:FEW_SHOT_K_MCQ]
FEW_SHOT_FREE = _free_short[:FEW_SHOT_K_FREE]

# Option B — hand-crafted multi-answer exemplars. Used for free-form questions
# that have >= 2 [ANS] placeholders, since the public-set few-shot pool filters
# those out (len(a) <= 2 in _short) and the model needs to see the comma-in-one-
# box format explicitly to keep Case-D dropouts in extract_free_answers low.
MULTI_ANSWER_EXEMPLARS = [
    {
        "question": (
            "A right triangle has legs of length 3 and 4. "
            "Its hypotenuse is [ANS] and its area is [ANS]."
        ),
        "answer": ["5", "6"],
    },
    {
        "question": (
            "Solve the system x + y = 7, x - y = 3. "
            "Then x = [ANS], y = [ANS], and x*y = [ANS]."
        ),
        "answer": ["5", "2", "10"],
    },
]

def _format_choices(options):
    labels = [chr(65 + i) for i in range(len(options))]
    return "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))

def _exemplar_block(items, is_mcq):
    if not items: return ""
    out = ["Worked examples (study the format, then solve the new question):\n"]
    for ex in items:
        q = ex["question"].strip()
        if is_mcq:
            out.append(f"Example question:\n{q}\n\nOptions:\n{_format_choices(ex['options'])}\n\nFinal answer: \\boxed{{{str(ex['answer']).strip().upper()}}}\n")
        else:
            a = ex["answer"] if isinstance(ex["answer"], list) else [ex["answer"]]
            out.append(f"Example question:\n{q}\n\nFinal answer: \\boxed{{{', '.join(str(x) for x in a)}}}\n")
    out.append("Now solve the NEW question below. Show your reasoning then finish with \\boxed{...}.\n")
    return "\n".join(out)

def _n_ans(question_text: str) -> int:
    return question_text.count("[ANS]")

def build_messages(item, hints: Optional[List[str]] = None):
    is_mcq = bool(item.get("options"))
    system = SYSTEM_PROMPT_MCQ if is_mcq else SYSTEM_PROMPT_MATH
    if is_mcq:
        exemplars = FEW_SHOT_MCQ
    else:
        # Option B: pick multi-answer exemplars when the question has >= 2 [ANS] slots.
        exemplars = MULTI_ANSWER_EXEMPLARS if _n_ans(item.get("question", "")) >= 2 else FEW_SHOT_FREE
    block  = _exemplar_block(exemplars, is_mcq)
    parts  = [block] if block else []
    parts.append("New question:\n" + item["question"].strip())
    if is_mcq:
        parts.append("\nOptions:\n" + _format_choices(item["options"]))
    if hints:
        parts.append(
            f"\nHint: previous attempts produced the candidate answer(s) {{{', '.join(str(h) for h in hints)}}}. "
            "Reconsider from scratch and either confirm or correct. Final answer inside \\boxed{}."
        )
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": "\n".join(parts)},
    ]

# Option D helper — re-label each [ANS] as [ANS_k] so subpart prompts can address slot k.
def _enumerate_ans_slots(question: str) -> str:
    parts = question.split("[ANS]")
    n = len(parts) - 1
    if n <= 0: return question
    out = parts[0]
    for k in range(1, n + 1):
        out += f"[ANS_{k}]" + parts[k]
    return out

def build_messages_for_slot(item, slot_idx: int):
    """Prompt the model to answer exactly one slot of a multi-[ANS] question."""
    q_enum = _enumerate_ans_slots(item["question"].strip())
    n = _n_ans(item.get("question", ""))
    user = (
        f"Question (it has {n} answer slots labeled [ANS_1] through [ANS_{n}]):\n"
        f"{q_enum}\n\n"
        f"Solve only [ANS_{slot_idx}]. Output exactly one value inside a single \\boxed{{...}}."
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT_SLOT},
        {"role": "user",   "content": user},
    ]

def chat_template_prompt(tok, messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


## 4. Answer extraction + voting + judging

In [ ]:
_LETTER_RE = re.compile(r"\\boxed\{\s*([A-Za-z])\s*\}")

def extract_letter(text: str) -> str:
    m = _LETTER_RE.search(text or "")
    if m: return m.group(1).upper()
    for line in reversed((text or "").splitlines()):
        ms = re.findall(r"\b([A-Z])\b", line.strip())
        if ms: return ms[-1]
    return ""

def extract_boxed_inner(text: str) -> str:
    if not text: return ""
    b = last_boxed_only_string(text)
    return (remove_boxed(b) or "").strip() if b else ""

def normalize_free_answer(text: str) -> str:
    s = extract_boxed_inner(text)
    if not s: return ""
    for r in (" ", "$", "\\,", "\\;", "\\!"):
        s = s.replace(r, "")
    return s

# ── Multi-[ANS] aware extraction ────────────────────────────────────────────
def n_ans_slots(item) -> int:
    """Number of answer slots: 1 for MCQ/single, else count of [ANS] in question."""
    if item.get("options"):
        return 1
    n = item.get("question", "").count("[ANS]")
    return n if n >= 1 else 1

def _strip_thinking(text: str) -> str:
    """Drop everything up to and including the last </think>, matching judger."""
    if not text: return ""
    j = text.rfind("</think>")
    return text[j + len("</think>"):] if j >= 0 else text

def _all_boxed_in_order(text: str) -> List[str]:
    """Every \\boxed{...} content in left-to-right order, brace-balanced."""
    out, i = [], 0
    needle = "\\boxed{"
    while True:
        j = text.find(needle, i)
        if j < 0: break
        k = j + len(needle)
        depth = 1
        while k < len(text) and depth > 0:
            if text[k] == '{': depth += 1
            elif text[k] == '}': depth -= 1
            k += 1
        if depth == 0:
            out.append(text[j + len(needle):k - 1].strip())
        i = k
    return out

def extract_free_answers(text: str, n_slots: int) -> Optional[List[str]]:
    """Return a list of length n_slots, or None if we can't reconcile."""
    post = _strip_thinking(text) or text
    boxes = _all_boxed_in_order(post) or _all_boxed_in_order(text)
    if not boxes:
        return None
    if n_slots == 1:
        return [boxes[-1]]
    # Case A: a single box with comma-separated answers matching n_slots
    if len(boxes) == 1:
        parts = judger.split_by_comma(boxes[0])
        if len(parts) == n_slots:
            return [p.strip() for p in parts]
        return None
    # Case B: exactly n_slots boxes -> in order
    if len(boxes) == n_slots:
        return boxes
    # Case C: more boxes than slots -> last n (those tend to be per-subpart finals)
    if len(boxes) > n_slots:
        return boxes[-n_slots:]
    # Case D: fewer boxes than slots -> try expanding comma-form boxes
    expanded = []
    for b in boxes:
        parts = judger.split_by_comma(b)
        if len(parts) > 1:
            expanded.extend(p.strip() for p in parts)
        else:
            expanded.append(b)
    if len(expanded) == n_slots:
        return expanded
    if len(expanded) > n_slots:
        return expanded[-n_slots:]
    return None

def _vote_key(s: str) -> str:
    """Canonicalize a single answer for vote bucketing (numeric-aware)."""
    if s is None: return ""
    t = s.strip().strip("$").strip()
    if not t: return ""
    # plain float
    try:
        v = float(t.replace(",", ""))
        return "0" if v == 0 else f"{v:.6g}"
    except Exception:
        pass
    # judger-normalized float
    try:
        norm = judger.norm_math_str(t)
        v = float(norm)
        return "0" if v == 0 else f"{v:.6g}"
    except Exception:
        pass
    try:
        return judger.norm_math_str(t)
    except Exception:
        return t

def vote(traces: List[str], item) -> Tuple[str, float]:
    """Return (canonical_answer_str, confidence_in_[0,1])."""
    if not traces: return "", 0.0
    is_mcq = bool(item.get("options"))
    if is_mcq:
        c = Counter()
        for t in traces:
            l = extract_letter(t)
            if l: c[l] += 1
        if not c: return "", 0.0
        top, cnt = c.most_common(1)[0]
        return top, cnt / len(traces)

    n = n_ans_slots(item)
    per_pos = [Counter() for _ in range(n)]
    rep     = [dict() for _ in range(n)]
    used = 0
    for t in traces:
        ans = extract_free_answers(t, n)
        if ans is None or len(ans) != n: continue
        used += 1
        for i, a in enumerate(ans):
            k = _vote_key(a)
            if not k: continue
            per_pos[i][k] += 1
            if k not in rep[i] or len(a) < len(rep[i][k]):
                rep[i][k] = a
    if used == 0: return "", 0.0
    parts, fracs = [], []
    for i in range(n):
        if not per_pos[i]: return "", 0.0
        top_k, top_n = per_pos[i].most_common(1)[0]
        parts.append(rep[i][top_k])
        fracs.append(top_n / used)
    return ", ".join(p.strip() for p in parts), min(fracs)

def judge_response(item, response: str) -> bool:
    if bool(item.get("options")):
        return extract_letter(response) == str(item["answer"]).strip().upper()
    gold = item["answer"] if isinstance(item["answer"], list) else [item["answer"]]
    try:
        return bool(judger.auto_judge(pred=response, gold=gold, options=[[]] * len(gold)))
    except Exception:
        return False

## 5. Build the external SFT corpus (checkpointed)

Downloads each HF dataset, converts every row to a `{prompt, completion}` line using the same chat template the model will see at inference, and streams them to `training/external_sft.jsonl`. If the file already exists this cell is a no-op (delete the file to rebuild).

In [7]:
def _make_freeform_item(question_text: str) -> dict:
    return {"question": question_text, "options": None}

def _row_to_pair(tok, hf_id: str, row: dict) -> Optional[Tuple[str, str]]:
    """Map any of the supported corpora to (prompt, completion)."""
    q, sol = None, None
    if hf_id == "AI-MO/NuminaMath-CoT":
        q   = row.get("problem")
        sol = row.get("solution")
    elif hf_id == "meta-math/MetaMathQA":
        q   = row.get("query")
        sol = row.get("response")
    elif hf_id == "hendrycks/competition_math":
        q   = row.get("problem")
        sol = row.get("solution")
    elif hf_id == "openai/gsm8k":
        q   = row.get("question")
        a   = row.get("answer", "")
        # GSM8K solutions end with "#### N" — rewrite to a boxed final answer.
        if "####" in a:
            reasoning, _, final = a.rpartition("####")
            sol = reasoning.strip() + f"\n\nFinal answer: \\boxed{{{final.strip()}}}"
        else:
            sol = a
    elif hf_id == "nvidia/OpenMathInstruct-2":
        q   = row.get("problem") or row.get("question")
        sol = row.get("generated_solution") or row.get("solution")
        ans = row.get("expected_answer")
        if sol and ans and "\\boxed" not in sol:
            sol = sol.rstrip() + f"\n\nFinal answer: \\boxed{{{ans}}}"
    elif hf_id == "TIGER-Lab/MathInstruct":
        q   = row.get("instruction")
        sol = row.get("output")
    elif hf_id == "nvidia/OpenMathReasoning":
        q   = row.get("problem")
        sol = row.get("generated_solution") or row.get("solution")
    if not q or not sol: return None
    if "\\boxed" not in sol:
        return None  # require a boxed answer so the model learns the output format
    prompt = chat_template_prompt(tok, build_messages(_make_freeform_item(q)))
    return prompt, sol.strip()

if RUN_BUILD_EXT_SFT and not EXT_SFT_PATH.exists():
    from datasets import load_dataset
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    n_written = 0
    tmp_path = EXT_SFT_PATH.with_suffix(".jsonl.tmp")
    with open(tmp_path, "w") as f:
        for hf_id, cfg, max_rows in EXT_DATASETS:
            print(f"→ {hf_id} (config={cfg}, cap={max_rows})")
            try:
                ds = load_dataset(hf_id, cfg, split="train", streaming=True)
            except Exception as e:
                print("  skipped:", e); continue
            kept = 0
            for i, row in enumerate(ds):
                if max_rows and i >= max_rows: break
                pair = _row_to_pair(tok, hf_id, row)
                if not pair: continue
                f.write(json.dumps({"prompt": pair[0], "completion": pair[1], "src": hf_id}) + "\n")
                kept += 1; n_written += 1
                if kept % 5000 == 0:
                    print(f"    {kept} rows")
            print(f"  kept {kept}")
    tmp_path.replace(EXT_SFT_PATH)
    print(f"Wrote {n_written} rows -> {EXT_SFT_PATH}")
else:
    print("External SFT corpus:", EXT_SFT_PATH, "present" if EXT_SFT_PATH.exists() else "SKIPPED")

External SFT corpus: training/external_sft.jsonl present


## 5b. Build the MCQ SFT corpus (checkpointed)

Downloads MMLU math/logic/physics subsets and converts each row to a
`{prompt, completion}` pair where the prompt is built with
`build_messages({question, options})` (MCQ system prompt + few-shot) and the
completion ends with `\boxed{LETTER}`. This is what teaches the SFT model to
emit the letter-form answer required for the competition's MCQ slice. Skipped
if `training/external_sft_mcq.jsonl` already exists.

In [ ]:
def _mcq_row_to_pair(tok, hf_id, cfg, row):
    if hf_id == "cais/mmlu":
        q       = row.get("question")
        choices = row.get("choices")
        ans_idx = row.get("answer")
        if not q or not choices or ans_idx is None: return None
        try:
            ai = int(ans_idx)
        except Exception:
            return None
        if ai < 0 or ai >= len(choices): return None
        letter = chr(65 + ai)
        item = {"question": q.strip(), "options": [str(c).strip() for c in choices]}
        prompt = chat_template_prompt(tok, build_messages(item))
        sol = (
            "I will analyze each option carefully.\n\n"
            f"The correct answer is option {letter}: {item['options'][ai]}.\n\n"
            f"\\boxed{{{letter}}}"
        )
        return prompt, sol
    return None

if RUN_BUILD_EXT_MCQ and not EXT_SFT_MCQ_PATH.exists():
    from datasets import load_dataset, concatenate_datasets
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    n_written = 0
    tmp_path = EXT_SFT_MCQ_PATH.with_suffix(".jsonl.tmp")
    with open(tmp_path, "w") as f:
        for hf_id, cfg, cap in EXT_DATASETS_MCQ:
            print(f"→ {hf_id} (config={cfg}, cap={cap})")
            # MMLU has no 'train' split; combine the non-test splits so we don't
            # train on the canonical test set. Falls back to whatever is available.
            ds = None
            for split in ("auxiliary_train", "dev+validation", "validation", "dev"):
                try:
                    ds = load_dataset(hf_id, cfg, split=split)
                    print(f"  using split={split}, rows={len(ds)}")
                    break
                except Exception:
                    continue
            if ds is None:
                try:
                    ds = load_dataset(hf_id, cfg, split="test")
                    print(f"  fallback to test split (no train available), rows={len(ds)}")
                except Exception as e:
                    print("  skipped:", e); continue
            kept = 0
            for i, row in enumerate(ds):
                if cap and i >= cap: break
                pair = _mcq_row_to_pair(tok, hf_id, cfg, row)
                if not pair: continue
                f.write(json.dumps({"prompt": pair[0], "completion": pair[1],
                                     "src": f"{hf_id}/{cfg}"}) + "\n")
                kept += 1; n_written += 1
            print(f"  kept {kept}")
    tmp_path.replace(EXT_SFT_MCQ_PATH)
    print(f"Wrote {n_written} MCQ rows -> {EXT_SFT_MCQ_PATH}")
else:
    print("MCQ SFT corpus:", EXT_SFT_MCQ_PATH,
          "present" if EXT_SFT_MCQ_PATH.exists() else "SKIPPED")


## 5c. Synthesize multi-[ANS] SFT data (checkpointed)

Pulls short single-answer problems from GSM8K and NuminaMath-CoT, groups
K ∈ [`MULTI_ANSWER_MIN_SLOTS`, `MULTI_ANSWER_MAX_SLOTS`] of them into a single
multi-part question with `[ANS]` markers, and writes
`{prompt, completion}` rows where the completion ends with
`\boxed{a, b, c}`. This directly attacks the multi-`[ANS]` format gap: the
public few-shot pool only contains single-answer items, so the SFT signal for
multi-slot output is otherwise zero. Skipped if
`training/multi_answer_sft.jsonl` already exists.

In [ ]:
def _extract_final_boxed(sol_text: str) -> Optional[str]:
    if not sol_text or "\\boxed{" not in sol_text: return None
    j = sol_text.rfind("\\boxed{")
    k = j + len("\\boxed{"); depth = 1
    while k < len(sol_text) and depth > 0:
        if   sol_text[k] == "{": depth += 1
        elif sol_text[k] == "}": depth -= 1
        k += 1
    if depth != 0: return None
    return sol_text[j + len("\\boxed{"):k - 1].strip()

if RUN_BUILD_MULTI_ANSWER and not MULTI_ANSWER_SFT_PATH.exists():
    from datasets import load_dataset
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    # Build a pool of (question, final_answer) pairs from short single-answer items.
    pool = []
    sources = [
        ("openai/gsm8k",         "main", None),
        ("AI-MO/NuminaMath-CoT", None,   30_000),
    ]
    for hf_id, cfg, cap in sources:
        try:
            ds = load_dataset(hf_id, cfg, split="train", streaming=True)
        except Exception as e:
            print(f"  pool skip {hf_id}: {e}"); continue
        kept = 0
        for i, row in enumerate(ds):
            if cap and i >= cap: break
            if hf_id == "openai/gsm8k":
                q = row.get("question") or ""
                a = row.get("answer") or ""
                if "####" not in a: continue
                final = a.rpartition("####")[2].strip()
            else:  # NuminaMath-CoT
                q = row.get("problem") or ""
                final = _extract_final_boxed(row.get("solution") or "")
                if not final: continue
            q = q.strip()
            if not q or not final: continue
            if len(q) > 280 or len(final) > 24: continue   # keep parts short
            if "[ANS]" in q: continue                      # don't recurse
            pool.append((q, final))
            kept += 1
        print(f"  pool from {hf_id}: +{kept}, total={len(pool)}")

    rng = random.Random(0)
    rng.shuffle(pool)

    tmp_path = MULTI_ANSWER_SFT_PATH.with_suffix(".jsonl.tmp")
    n_written = 0
    LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)"]
    with open(tmp_path, "w") as f:
        i = 0
        while n_written < N_MULTI_ANSWER and i + MULTI_ANSWER_MAX_SLOTS <= len(pool):
            k = rng.randint(MULTI_ANSWER_MIN_SLOTS, MULTI_ANSWER_MAX_SLOTS)
            group = pool[i:i + k]; i += k
            labels = LABELS[:k]
            parts_q = []
            for lbl, (q, _) in zip(labels, group):
                parts_q.append(f"{lbl} {q} [ANS]")
            combined_q = (
                "Solve each independent part. Write the final answer for each "
                "where indicated by [ANS], then collect all answers in order.\n\n"
                + "\n\n".join(parts_q)
            )
            ans_list = [a for _, a in group]
            item = {"question": combined_q, "options": None}
            prompt = chat_template_prompt(tok, build_messages(item))
            sol_lines = [f"For {lbl}, the answer is {a}." for lbl, (_, a) in zip(labels, group)]
            completion = (
                "I will solve each part independently.\n\n"
                + "\n".join(sol_lines)
                + f"\n\nFinal answer: \\boxed{{{', '.join(ans_list)}}}"
            )
            f.write(json.dumps({"prompt": prompt, "completion": completion,
                                 "src": "synthetic/multi-ans"}) + "\n")
            n_written += 1
    tmp_path.replace(MULTI_ANSWER_SFT_PATH)
    print(f"Wrote {n_written} synthetic multi-[ANS] rows -> {MULTI_ANSWER_SFT_PATH}")
else:
    print("Multi-[ANS] SFT corpus:", MULTI_ANSWER_SFT_PATH,
          "present" if MULTI_ANSWER_SFT_PATH.exists() else "SKIPPED")


## 6. QLoRA SFT (resumable)

HuggingFace `Trainer` checkpoints every `SFT_SAVE_STEPS` to `training/sft_lora/checkpoint-N`. Re-running this cell after a crash auto-resumes from the latest checkpoint.

In [ ]:
def _latest_ckpt(dir_path: Path) -> Optional[str]:
    ckpts = sorted(glob.glob(str(dir_path / "checkpoint-*")),
                    key=lambda p: int(p.rsplit("-", 1)[-1]))
    return ckpts[-1] if ckpts else None

if RUN_SFT:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig
    from datasets import load_dataset

    # Load every SFT corpus we have on disk: math single-answer + MCQ + synthetic
    # multi-[ANS]. Each corpus targets a different failure mode of the base model.
    data_files = []
    if EXT_SFT_PATH.exists():          data_files.append(str(EXT_SFT_PATH))
    if EXT_SFT_MCQ_PATH.exists():      data_files.append(str(EXT_SFT_MCQ_PATH))
    if MULTI_ANSWER_SFT_PATH.exists(): data_files.append(str(MULTI_ANSWER_SFT_PATH))
    assert data_files, "No SFT corpus on disk — run sections 5 / 5b / 5c first."
    print("SFT data files:", data_files)
    ds = load_dataset("json", data_files=data_files, split="train")
    ds = ds.shuffle(seed=0)
    print(f"SFT total rows: {len(ds)}")

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                              bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    base = prepare_model_for_kbit_training(base)

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                          "gate_proj","up_proj","down_proj"],
    )

    eos = tok.eos_token
    def fmt(ex): return {"text": ex["prompt"] + ex["completion"] + eos}
    ds = ds.map(fmt, remove_columns=[c for c in ds.column_names if c not in ("prompt","completion")])

    cfg = SFTConfig(
        output_dir=str(SFT_LORA_DIR),
        per_device_train_batch_size=SFT_BSZ,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        num_train_epochs=SFT_EPOCHS,
        learning_rate=SFT_LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
        bf16=True, logging_steps=SFT_LOG_STEPS,
        save_strategy="steps", save_steps=SFT_SAVE_STEPS, save_total_limit=3,
        max_length=2048, packing=True, report_to="none",
        gradient_checkpointing=True, optim="adamw_8bit",
        dataset_text_field="text",
    )
    trainer = SFTTrainer(model=base, args=cfg, train_dataset=ds,
                          peft_config=lora_cfg, processing_class=tok)
    resume = _latest_ckpt(SFT_LORA_DIR)
    print("Resuming from", resume) if resume else print("Starting SFT fresh")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(SFT_LORA_DIR))
    print("SFT-LoRA saved ->", SFT_LORA_DIR)
    del trainer, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping SFT.")


## 7. Merge SFT adapter into BF16 weights (for vLLM)

In [9]:
if RUN_MERGE_SFT and not MERGED_SFT_DIR.exists():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    print("Merging SFT adapter…")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, str(SFT_LORA_DIR)).merge_and_unload()
    merged.save_pretrained(MERGED_SFT_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True).save_pretrained(MERGED_SFT_DIR)
    del base, merged; gc.collect(); torch.cuda.empty_cache()
    print("Merged ->", MERGED_SFT_DIR)
else:
    print("Merged SFT:", MERGED_SFT_DIR, "present" if MERGED_SFT_DIR.exists() else "skipped")

Merged SFT: /tmp/prgowda/151b_training/merged_sft present


## 8. Self-sample the SFT model on the public set (checkpointed)

Writes `training/self_samples.jsonl` with one line per `(id, sample_idx)`. Re-running resumes from where it stopped (dedup on `(id, sample_idx)`).

In [10]:
def _read_done_keys(path: Path, key_fields=("id", "sample_idx")):
    if not path.exists(): return set()
    done = set()
    with open(path) as f:
        for line in f:
            try:
                r = json.loads(line)
                done.add(tuple(r[k] for k in key_fields))
            except Exception: pass
    return done

if RUN_SELF_SAMPLE:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_path = str(MERGED_SFT_DIR) if MERGED_SFT_DIR.exists() else BASE_MODEL_ID
    use_bnb = (model_path == BASE_MODEL_ID)
    print("Self-sampling with:", model_path)

    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    done = _read_done_keys(SELF_SAMPLES_PATH)
    have_per_id = Counter(k[0] for k in done)
    todo = [it for it in public_data if have_per_id[it["id"]] < N_SAMPLES_SELF]
    print(f"To sample: {len(todo)}/{len(public_data)} questions (already have {len(done)} traces)")

    if todo:
        kw = dict(model=model_path, trust_remote_code=True,
                   gpu_memory_utilization=0.80, max_model_len=INFER_MAX_MODEL_LEN,
                   max_num_seqs=256, enable_prefix_caching=True)
        if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")
        llm = LLM(**kw)
        sp = SamplingParams(n=N_SAMPLES_SELF, max_tokens=MAX_NEW_TOKENS_SAMPLE,
                             temperature=0.9, top_p=0.95, top_k=40)
        # process in chunks to checkpoint frequently
        for start in range(0, len(todo), INFER_CHUNK):
            batch = todo[start:start+INFER_CHUNK]
            prompts = [chat_template_prompt(tok, build_messages(it)) for it in batch]
            outs = llm.generate(prompts, sampling_params=sp)
            with open(SELF_SAMPLES_PATH, "a") as f:
                for item, out in zip(batch, outs):
                    for i, c in enumerate(out.outputs):
                        if (item["id"], i) in done: continue
                        f.write(json.dumps({
                            "id": item["id"], "sample_idx": i,
                            "is_mcq": bool(item.get("options")),
                            "gold": item.get("answer"),
                            "trace": c.text.strip(),
                        }) + "\n")
            print(f"  flushed up to question {start+len(batch)}/{len(todo)}")
        del llm; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception: pass
else:
    print("Skipping self-sample.")

Skipping self-sample.


## 9. Build DPO pairs from the self-samples (idempotent)

For each public question, pair its longest correct trace as `chosen` with its longest incorrect trace as `rejected`.

In [11]:
if RUN_BUILD_DPO:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    id_to_item = {it["id"]: it for it in public_data}
    bucket = defaultdict(list)
    with open(SELF_SAMPLES_PATH) as f:
        for line in f:
            r = json.loads(line); bucket[r["id"]].append(r)
    n_pairs = 0
    with open(DPO_PAIRS_PATH, "w") as out:
        for qid, rows in bucket.items():
            item = id_to_item.get(qid)
            if not item: continue
            good, bad = [], []
            for r in rows:
                (good if judge_response(item, r["trace"]) else bad).append(r["trace"])
            if not good or not bad: continue
            chosen   = max(good, key=len)
            rejected = max(bad,  key=len)
            prompt = chat_template_prompt(tok, build_messages(item))
            out.write(json.dumps({"prompt": prompt, "chosen": chosen,
                                    "rejected": rejected, "id": qid}) + "\n")
            n_pairs += 1
    print(f"DPO pairs: {n_pairs} -> {DPO_PAIRS_PATH}")
else:
    print("Skipping DPO-pair build.")

/home/prgowda/private/151B_SP26_Competition-main/.venv/lib/python3.14/site-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


DPO pairs: 404 -> training/dpo_pairs.jsonl


## 10. DPO training on top of SFT (resumable)

In [12]:
if RUN_DPO:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, PeftModel
    from trl import DPOTrainer, DPOConfig
    from datasets import load_dataset

    assert DPO_PAIRS_PATH.exists(), "Run section 9 first."
    ds = load_dataset("json", data_files=str(DPO_PAIRS_PATH), split="train")

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                              bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    if SFT_LORA_DIR.exists():
        policy = PeftModel.from_pretrained(base, str(SFT_LORA_DIR), is_trainable=True)
        peft_cfg = None
    else:
        policy = base
        peft_cfg = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
                                task_type="CAUSAL_LM",
                                target_modules=["q_proj","k_proj","v_proj","o_proj",
                                                  "gate_proj","up_proj","down_proj"])

    cfg = DPOConfig(
        output_dir=str(DPO_LORA_DIR),
        per_device_train_batch_size=1, gradient_accumulation_steps=16,
        num_train_epochs=DPO_EPOCHS, learning_rate=DPO_LR, beta=DPO_BETA,
        bf16=True, logging_steps=10,
        save_strategy="steps", save_steps=100, save_total_limit=3,
        # max_length=MAX_MODEL_LEN, max_prompt_length=MAX_MODEL_LEN // 2,
        report_to="none", gradient_checkpointing=True, optim="paged_adamw_8bit",
    )
    trainer = DPOTrainer(model=policy, ref_model=None, args=cfg, train_dataset=ds,
                          processing_class=tok, peft_config=peft_cfg)
    resume = _latest_ckpt(DPO_LORA_DIR)
    print("Resuming DPO from", resume) if resume else print("Starting DPO fresh")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(DPO_LORA_DIR))
    del trainer, policy, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping DPO.")

Generating train split: 0 examples [00:00, ? examples/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/home/prgowda/private/151B_SP26_Competition-main/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Adding EOS to train dataset:   0%|          | 0/404 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/404 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Resuming DPO from training/dpo_lora/checkpoint-26


Step,Training Loss


## 11. (Optional) GRPO with the project Judger as outcome reward

Uses `trl.GRPOTrainer`. The reward function returns 1.0 if the completion's extracted answer is judged correct, else 0.0. Resumable via Trainer checkpoints.

In [13]:
if RUN_GRPO:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, PeftModel
    from trl import GRPOTrainer, GRPOConfig
    from datasets import Dataset

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    # Build a small prompt dataset (public set) keyed to gold answers.
    rows = []
    for it in public_data:
        prompt = chat_template_prompt(tok, build_messages(it))
        rows.append({"prompt": prompt,
                       "gold": json.dumps(it.get("answer")),
                       "is_mcq": bool(it.get("options"))})
    ds = Dataset.from_list(rows)

    def reward_fn(prompts, completions, **kwargs):
        golds   = kwargs["gold"]
        is_mcqs = kwargs["is_mcq"]
        out = []
        for comp, gold_json, is_mcq in zip(completions, golds, is_mcqs):
            gold = json.loads(gold_json)
            item = {"answer": gold, "options": ["x"] if is_mcq else None}
            out.append(1.0 if judge_response(item, comp) else 0.0)
        return out

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                              bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    start_adapter = DPO_LORA_DIR if DPO_LORA_DIR.exists() else SFT_LORA_DIR
    if start_adapter.exists():
        policy = PeftModel.from_pretrained(base, str(start_adapter), is_trainable=True)
        peft_cfg = None
    else:
        policy = base
        peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                                task_type="CAUSAL_LM",
                                target_modules=["q_proj","k_proj","v_proj","o_proj"])

    cfg = GRPOConfig(
        output_dir=str(GRPO_LORA_DIR),
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=GRPO_EPOCHS, learning_rate=GRPO_LR, beta=GRPO_BETA,
        num_generations=GRPO_NUM_GENERATIONS, max_completion_length=MAX_NEW_TOKENS_SAMPLE,
        max_prompt_length=MAX_MODEL_LEN // 2, temperature=0.9,
        use_vllm=True, vllm_gpu_memory_utilization=0.4,
        bf16=True, logging_steps=5,
        save_strategy="steps", save_steps=50, save_total_limit=3,
        report_to="none", gradient_checkpointing=True, optim="paged_adamw_8bit",
    )
    trainer = GRPOTrainer(model=policy, reward_funcs=reward_fn,
                            args=cfg, train_dataset=ds,
                            processing_class=tok, peft_config=peft_cfg)
    resume = _latest_ckpt(GRPO_LORA_DIR)
    print("Resuming GRPO from", resume) if resume else print("Starting GRPO fresh")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(GRPO_LORA_DIR))
    del trainer, policy, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping GRPO.")

Skipping GRPO.


## 12. Merge final adapter (GRPO > DPO > SFT, whichever is latest)

In [14]:
if RUN_MERGE_FINAL and not MERGED_FINAL_DIR.exists():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    adapter = (GRPO_LORA_DIR if GRPO_LORA_DIR.exists()
               else DPO_LORA_DIR if DPO_LORA_DIR.exists()
               else SFT_LORA_DIR)
    assert adapter.exists(), "No adapter found."
    print("Merging final adapter:", adapter)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, str(adapter)).merge_and_unload()
    merged.save_pretrained(MERGED_FINAL_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True).save_pretrained(MERGED_FINAL_DIR)
    del base, merged; gc.collect(); torch.cuda.empty_cache()
    print("Merged ->", MERGED_FINAL_DIR)
else:
    print("Merged final:", MERGED_FINAL_DIR, "present" if MERGED_FINAL_DIR.exists() else "skipped")

Merged final: /tmp/prgowda/151b_training/merged_final present


## 13. Private inference with self-consistency (checkpointed in chunks)

Generates `N_SAMPLES_PER_Q` traces per question, appending to `results/private_samples.jsonl` after every chunk. On resume, questions that already have enough samples are skipped.

In [ ]:
def _load_samples(path: Path):
    by_id = defaultdict(list)
    if not path.exists(): return by_id
    with open(path) as f:
        for line in f:
            try:
                r = json.loads(line); by_id[r["id"]].append(r)
            except Exception: pass
    return by_id

if RUN_PRIVATE_INFER:
    import os
    os.environ["VLLM_USE_DEEP_GEMM"]    = "0"
    os.environ["VLLM_DEEP_GEMM_WARMUP"] = "skip"
    os.environ["VLLM_USE_V1"] = "0"

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_path = str(MERGED_FINAL_DIR) if MERGED_FINAL_DIR.exists() else (
        str(MERGED_SFT_DIR) if MERGED_SFT_DIR.exists() else BASE_MODEL_ID)
    use_bnb = (model_path == BASE_MODEL_ID)
    print("Private inference with:", model_path)

    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    by_id = _load_samples(PRIVATE_SAMPLES_PATH)

    # Build per-pass budgets. With stratified sampling we split into a precision
    # pass (low T) and an exploration pass (high T); legacy mode is a single pass.
    if STRATIFIED_SAMPLING:
        n_precision   = N_SAMPLES_PER_Q // 2
        n_exploration = N_SAMPLES_PER_Q - n_precision
        PASSES = [
            ("precision",   n_precision,   INFER_T_PRECISION),
            ("exploration", n_exploration, INFER_T_EXPLORATION),
        ]
    else:
        PASSES = [("legacy", N_SAMPLES_PER_Q, INFER_TEMPERATURE)]

    def _have_for_pass(item, pass_name):
        return sum(1 for r in by_id.get(item["id"], []) if r.get("pass") == pass_name)

    total_todo = 0
    plans = []
    for pass_name, n_target, temp in PASSES:
        todo = [(it, n_target - _have_for_pass(it, pass_name)) for it in private_data
                if _have_for_pass(it, pass_name) < n_target]
        plans.append((pass_name, temp, todo))
        total_todo += len(todo)
    print(f"Per-pass items needing samples: {[(p, len(t)) for p, _, t in plans]}  total={total_todo}")

    if total_todo:
        kw = dict(model=model_path, trust_remote_code=True,
                   gpu_memory_utilization=0.80, max_model_len=INFER_MAX_MODEL_LEN,
                   max_num_seqs=128,
                   max_num_batched_tokens=16384,
                   enable_prefix_caching=True,
                   enforce_eager=True)
        if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")

        import gc, torch
        gc.collect(); torch.cuda.empty_cache()
        llm = LLM(**kw)

        for pass_name, temp, todo in plans:
            if not todo:
                print(f"  [{pass_name} T={temp}] up-to-date.")
                continue
            n_max = max(need for _, need in todo)
            sp = SamplingParams(n=n_max, max_tokens=MAX_NEW_TOKENS_INFER,
                                 temperature=temp, top_p=INFER_TOP_P, top_k=INFER_TOP_K)
            print(f"  [{pass_name} T={temp}] generating up to n={n_max} for {len(todo)} items")
            for start in range(0, len(todo), INFER_CHUNK):
                batch = todo[start:start+INFER_CHUNK]
                prompts = [chat_template_prompt(tok, build_messages(it)) for it, _ in batch]
                outs = llm.generate(prompts, sampling_params=sp)
                with open(PRIVATE_SAMPLES_PATH, "a") as f:
                    for (item, need), out in zip(batch, outs):
                        base_idx = len(by_id[item["id"]])
                        for j, c in enumerate(out.outputs[:need]):
                            rec = {"id": item["id"], "sample_idx": base_idx + j,
                                    "is_mcq": bool(item.get("options")),
                                    "pass": pass_name, "temperature": temp,
                                    "trace": c.text.strip()}
                            by_id[item["id"]].append(rec)
                            f.write(json.dumps(rec) + "\n")
                print(f"    [{pass_name}] flushed {start+len(batch)}/{len(todo)}")

        del llm; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception: pass
else:
    print("Skipping private inference.")


## 14. Vote, then progressive-hint round on low-confidence items (checkpointed)

In [ ]:
by_id = _load_samples(PRIVATE_SAMPLES_PATH)
voted = {}
for item in private_data:
    traces = [r["trace"] for r in by_id.get(item["id"], [])]
    key, frac = vote(traces, item)
    voted[item["id"]] = {"key": key, "frac": frac,
                          "is_mcq": bool(item.get("options"))}
low = [it for it in private_data if voted[it["id"]]["frac"] < PROGRESSIVE_HINT_TAU
       and voted[it["id"]]["key"]]
print(f"Low-confidence (<{PROGRESSIVE_HINT_TAU}): {len(low)}/{len(private_data)}")

if RUN_HINT_ROUND and low:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_path = str(MERGED_FINAL_DIR) if MERGED_FINAL_DIR.exists() else (
        str(MERGED_SFT_DIR) if MERGED_SFT_DIR.exists() else BASE_MODEL_ID)
    use_bnb = (model_path == BASE_MODEL_ID)
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    hint_by_id = _load_samples(PRIVATE_HINTS_PATH)
    todo = [it for it in low if len(hint_by_id[it["id"]]) < N_HINT_SAMPLES]
    print(f"Hint round todo: {len(todo)}")

    if todo:
        kw = dict(model=model_path, trust_remote_code=True,
                   gpu_memory_utilization=0.85, max_model_len=INFER_MAX_MODEL_LEN,
                   max_num_seqs=256, enable_prefix_caching=True)
        if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")
        llm = LLM(**kw)
        sp = SamplingParams(n=N_HINT_SAMPLES, max_tokens=MAX_NEW_TOKENS_INFER,
                             temperature=max(0.4, INFER_TEMPERATURE - 0.2),
                             top_p=INFER_TOP_P, top_k=INFER_TOP_K)
        for start in range(0, len(todo), INFER_CHUNK):
            batch = todo[start:start+INFER_CHUNK]
            prompts = [chat_template_prompt(tok, build_messages(it, hints=[voted[it["id"]]["key"]]))
                       for it in batch]
            outs = llm.generate(prompts, sampling_params=sp)
            with open(PRIVATE_HINTS_PATH, "a") as f:
                for item, out in zip(batch, outs):
                    base_idx = len(hint_by_id[item["id"]])
                    for j, c in enumerate(out.outputs):
                        rec = {"id": item["id"], "sample_idx": base_idx + j,
                                "is_mcq": bool(item.get("options")),
                                "trace": c.text.strip()}
                        hint_by_id[item["id"]].append(rec)
                        f.write(json.dumps(rec) + "\n")
            print(f"  hint flushed {start+len(batch)}/{len(todo)}")
        del llm; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception: pass

# Re-vote, mixing in hint samples for low-confidence items.
hint_by_id = _load_samples(PRIVATE_HINTS_PATH)
for item in private_data:
    combined = [r["trace"] for r in by_id.get(item["id"], [])] + \
                 [r["trace"] for r in hint_by_id.get(item["id"], [])]
    if not combined: continue
    key, frac = vote(combined, item)
    voted[item["id"]] = {"key": key, "frac": frac,
                          "is_mcq": bool(item.get("options"))}
print("Voting refreshed with hint samples.")

## 14b. Per-subpart fallback for multi-[ANS] failures (option D)

For multi-[ANS] questions where the voted answer is empty, has the wrong number
of comma-separated parts, or the slot-wise confidence is below
`SUBPART_CONF_THRESHOLD`, ask the model each slot independently using
`build_messages_for_slot`. Vote per slot and reassemble. Resumable via
`results/private_subpart_samples.jsonl`.

In [ ]:
def _needs_subpart(item):
    if item.get("options"): return False
    n = item.get("question", "").count("[ANS]")
    if n < 2: return False
    v = voted.get(item["id"], {})
    key = v.get("key") or ""
    if not key:
        return True
    parts = [p.strip() for p in key.split(",")]
    if len(parts) != n:
        return True
    if v.get("frac", 0.0) < SUBPART_CONF_THRESHOLD:
        return True
    return False

def _load_subpart_samples(path: Path):
    by_key = defaultdict(list)
    if not path.exists(): return by_key
    with open(path) as f:
        for line in f:
            try:
                r = json.loads(line); by_key[(r["id"], r["slot"])].append(r["trace"])
            except Exception: pass
    return by_key

if RUN_SUBPART_FALLBACK:
    failing = [it for it in private_data if _needs_subpart(it)]
    print(f"Subpart fallback candidates: {len(failing)}/{len(private_data)}")

    sub_by_key = _load_subpart_samples(PRIVATE_SUBPART_PATH)
    work = []
    for it in failing:
        n = it["question"].count("[ANS]")
        for k in range(1, n + 1):
            need = N_SUBPART_SAMPLES - len(sub_by_key[(it["id"], k)])
            if need > 0:
                work.append((it, k, n, need))
    print(f"Subpart prompts to run: {len(work)}")

    if work:
        import os
        os.environ["VLLM_USE_DEEP_GEMM"]    = "0"
        os.environ["VLLM_DEEP_GEMM_WARMUP"] = "skip"
        os.environ["VLLM_USE_V1"] = "0"
        from transformers import AutoTokenizer
        from vllm import LLM, SamplingParams

        model_path = str(MERGED_FINAL_DIR) if MERGED_FINAL_DIR.exists() else (
            str(MERGED_SFT_DIR) if MERGED_SFT_DIR.exists() else BASE_MODEL_ID)
        use_bnb = (model_path == BASE_MODEL_ID)
        tok_sp = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        tok_sp.pad_token = tok_sp.eos_token

        kw = dict(model=model_path, trust_remote_code=True,
                   gpu_memory_utilization=0.80, max_model_len=INFER_MAX_MODEL_LEN,
                   max_num_seqs=128, max_num_batched_tokens=16384,
                   enable_prefix_caching=True, enforce_eager=True)
        if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")
        import gc, torch
        gc.collect(); torch.cuda.empty_cache()
        llm = LLM(**kw)
        n_max = max(w[3] for w in work)
        sp = SamplingParams(n=n_max, max_tokens=MAX_NEW_TOKENS_INFER,
                             temperature=SUBPART_TEMP, top_p=INFER_TOP_P, top_k=INFER_TOP_K)
        for start in range(0, len(work), INFER_CHUNK):
            batch = work[start:start+INFER_CHUNK]
            prompts = [chat_template_prompt(tok_sp, build_messages_for_slot(it, k))
                       for it, k, _, _ in batch]
            outs = llm.generate(prompts, sampling_params=sp)
            with open(PRIVATE_SUBPART_PATH, "a") as f:
                for (it, k, n, need), out in zip(batch, outs):
                    for c in out.outputs[:need]:
                        rec = {"id": it["id"], "slot": k, "n_slots": n,
                                "trace": c.text.strip()}
                        sub_by_key[(it["id"], k)].append(c.text.strip())
                        f.write(json.dumps(rec) + "\n")
            print(f"  subpart flushed {start+len(batch)}/{len(work)}")
        del llm; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception: pass

    # Vote per slot and reassemble. Replace voted entry only if every slot has at
    # least one extracted answer; prefer the subpart result over the old vote when
    # the old vote was empty, had the wrong number of parts, or had lower minimum
    # confidence.
    def _slot_answer(traces):
        c = Counter(); rep = {}
        for t in traces:
            post = _strip_thinking(t) or t
            boxes = _all_boxed_in_order(post) or _all_boxed_in_order(t)
            if not boxes: continue
            a = boxes[-1].strip()
            key = _vote_key(a)
            if not key: continue
            c[key] += 1
            if key not in rep or len(a) < len(rep[key]):
                rep[key] = a
        if not c: return None, 0.0
        top_k, top_n = c.most_common(1)[0]
        return rep[top_k].strip(), top_n / max(1, sum(c.values()))

    n_recovered = 0
    for it in [x for x in private_data if not x.get("options") and x["question"].count("[ANS]") >= 2]:
        n = it["question"].count("[ANS]")
        slot_answers, slot_fracs = [], []
        ok = True
        for k in range(1, n + 1):
            traces = sub_by_key.get((it["id"], k), [])
            if not traces: ok = False; break
            a, frac = _slot_answer(traces)
            if a is None: ok = False; break
            slot_answers.append(a); slot_fracs.append(frac)
        if not ok: continue
        new_key  = ", ".join(slot_answers)
        new_frac = min(slot_fracs)
        v        = voted.get(it["id"], {})
        old_key  = v.get("key", "") or ""
        old_frac = v.get("frac", 0.0)
        old_parts = [p.strip() for p in old_key.split(",")] if old_key else []
        replace = (not old_key) or (len(old_parts) != n) or (new_frac > old_frac)
        if replace:
            voted[it["id"]] = {"key": new_key, "frac": new_frac, "is_mcq": False}
            n_recovered += 1
    print(f"Subpart fallback updated {n_recovered} items.")


## 15. Write `submission.csv`

In [ ]:
def finalize_response(v):
    key = v.get("key") or ""
    if not key:
        return "\\boxed{}"
    return f"\\boxed{{{key}}}"

with open(SUBMISSION_CSV, "w", newline="") as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(["id", "response"])
    for item in private_data:
        v = voted.get(item["id"], {"key": "", "is_mcq": bool(item.get("options"))})
        w.writerow([item["id"], finalize_response(v)])
print(f"Wrote {SUBMISSION_CSV} with {len(private_data)} rows.")

## 16. Public-set accuracy sanity check (optional)

In [ ]:
RUN_PUBLIC_EVAL = True
if RUN_PUBLIC_EVAL:
    bucket = defaultdict(list)
    if SELF_SAMPLES_PATH.exists():
        with open(SELF_SAMPLES_PATH) as f:
            for line in f:
                r = json.loads(line); bucket[r["id"]].append(r["trace"])
    n_ok = 0; n_done = 0
    for it in public_data:
        traces = bucket.get(it["id"], [])
        if not traces: continue
        key, frac = vote(traces, it)
        synth = finalize_response({"key": key, "is_mcq": bool(it.get("options"))})
        n_ok += int(judge_response(it, synth)); n_done += 1
    print(f"Public-set accuracy (from cached self-samples): {n_ok}/{n_done} = {100*n_ok/max(1,n_done):.2f}%")
else:
    print("Set RUN_PUBLIC_EVAL=True to estimate.")